# House Prices: Advanced Regression Techniques
**Autor:** [Tu Nombre]  
**Descripción:** Este notebook documenta el pipeline completo para la competencia de Kaggle sobre la predicción de precios de viviendas en Ames, Iowa.  
**Metodología:** El flujo de trabajo abarca desde la exploración inicial (EDA), ingeniería de características, hasta la implementación de un Stacking Regressor (Ridge + XGBoost + LightGBM).

---

In [ ]:
# ==============================================================================
# ENTORNO NATIVO DE KAGGLE E INSPECCIÓN DE ARCHIVOS
# ==============================================================================
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

print("=== RUTAS DE ARCHIVOS DISPONIBLES ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
print("=" * 50)

## Fase 1: Diagnóstico de la variable objetivo (SalePrice)
En esta etapa, evaluamos la distribución de nuestra variable objetivo. Un diagnóstico de asimetría (*skewness*) es fundamental, ya que los modelos de regresión lineal suelen desempeñarse mejor cuando la variable dependiente sigue una distribución cercana a la normal.

* **Acción:** Aplicación de `np.log1p` para normalizar la distribución y reducir el impacto de valores extremos.
* **Métricas:** Comparación de *Skewness* y *Kurtosis* antes y después de la transformación.

In [ ]:
# ==============================================================================
# FASE 1 - PASO A: CARGA Y DIAGNÓSTICO DE LA VARIABLE OBJETIVO
# ==============================================================================

# 1. Carga de datos usando las rutas verificadas
train = pd.read_csv('../input/competitions/home-data-for-ml-course/train.csv')
test = pd.read_csv('../input/competitions/home-data-for-ml-course/test.csv')

# 2. Aislar variable objetivo y calcular su transformación logarítmica
target = train['SalePrice']
target_log = np.log1p(target)

# 3. Imprimir métricas de asimetría y concentración (Antes y Después)
print("\n=== DIAGNÓSTICO MATEMÁTICO ===")
print("Métricas Originales:")
print(f"  -> Sesgo (Skewness):        {skew(target):.2f}")
print(f"  -> Concentración (Kurtosis): {kurtosis(target):.2f}")
print("-" * 50)
print("Métricas Tras Transformación Logarítmica [np.log1p()]:")
print(f"  -> Nuevo Sesgo:             {skew(target_log):.2f}")
print(f"  -> Nueva Concentración:     {kurtosis(target_log):.2f}")
print("==================================================")

# 4. Graficar la distribución original vs transformada
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(target, kde=True, ax=ax[0], color='blue')
ax[0].set_title('Distribución Original de SalePrice')
ax[0].set_xlabel('SalePrice (Dólares Reales)')

sns.histplot(target_log, kde=True, ax=ax[1], color='green')
ax[1].set_title('Distribución Tras Transformación Logarítmica log(1+x)')
ax[1].set_xlabel('SalePrice (Escala Logarítmica)')

plt.tight_layout()
plt.show()

# 5. Conocer la estructura de tipos de variables en el dataset
print("\n=== TAXONOMÍA DE VARIABLES EN X ===")
print(train.drop(['Id', 'SalePrice'], axis=1).dtypes.value_counts())

# ==============================================================================
# FASE 1: EXPLORACIÓN Y DIAGNÓSTICO (EDA)
# Paso B: Auditoría de Datos Faltantes (Missing Values)
# ==============================================================================

# 1. Calcular el total de nulos y el porcentaje correspondiente por columna
missing_total = train.isnull().sum()
missing_percent = (train.isnull().sum() / len(train)) * 100
# 2. Consolidar los resultados en un DataFrame estructurado
missing_report = pd.DataFrame({
    'Total Nulos': missing_total, 
    'Porcentaje (%)': missing_percent
})

# 3. Filtrar para mostrar únicamente las variables que tienen nulos y ordenar descendentemente
missing_report = missing_report[missing_report['Total Nulos'] > 0].sort_values(by='Total Nulos', ascending=False)

print("\n==================================================")
print("   FASE 1 - PASO B: COLUMNAS CON DATOS FALTANTES")
print("==================================================")
if not missing_report.empty:
    # Mostramos las 20 variables con mayor cantidad de vacíos
    print(missing_report.head(20))  
else:
    print("¡Excelente! No se detectaron datos faltantes en este dataset.")
print("==================================================")

## Fase 2: Preprocesamiento (Data Pipeline)
En esta fase, construimos un pipeline robusto de limpieza de datos para garantizar la consistencia entre los conjuntos de entrenamiento y prueba.

* **Imputación Estructural:** Manejo de datos faltantes en variables de garaje, sótanos y características categóricas, reemplazándolos con valores lógicos ("None" o 0).
* **Imputación por Vecindario:** Uso de medianas calculadas exclusivamente en el set de entrenamiento para evitar el *data leakage* al tratar `LotFrontage`.
* **Codificación:** Aplicación de un diccionario centralizado para variables ordinales y *One-Hot Encoding* para variables nominales, asegurando una estructura matemática uniforme para el modelo.

In [ ]:
# ==============================================================================
# FASE 2 — PREPROCESAMIENTO (DATA PIPELINE)
# Paso 2A: Imputación Dirigida (Manejo de Datos Faltantes)
# ==============================================================================

# Hacemos copias de trabajo para no alterar los DataFrames originales en bruto
X_train = train.drop(['Id', 'SalePrice'], axis=1).copy()
X_test = test.drop(['Id'], axis=1).copy()

# ------------------------------------------------------------------------------
# 1. IMPUTACIÓN ESTRUCTURAL CATEGÓRICA -> Reemplazar con "None" (No tiene)
# ------------------------------------------------------------------------------
structural_categorical = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]

for col in structural_categorical:
    X_train[col] = X_train[col].fillna("None")
    X_test[col] = X_test[col].fillna("None")

# ------------------------------------------------------------------------------
# 2. IMPUTACIÓN ESTRUCTURAL NUMÉRICA -> Reemplazar con 0
# ------------------------------------------------------------------------------
# Si no hay garage o sótano, el año es 0 y el área de mampostería es 0
X_train['GarageYrBlt'] = X_train['GarageYrBlt'].fillna(0)
X_test['GarageYrBlt'] = X_test['GarageYrBlt'].fillna(0)

X_train['MasVnrArea'] = X_train['MasVnrArea'].fillna(0)
X_test['MasVnrArea'] = X_test['MasVnrArea'].fillna(0)

# ------------------------------------------------------------------------------
# 3. AUSENCIAS REALES CATEGÓRICAS -> Imputación por la Moda del entrenamiento
# ------------------------------------------------------------------------------
# Para 'Electrical' y cualquier nulo remanente menor en el set de test
X_train['Electrical'] = X_train['Electrical'].fillna(X_train['Electrical'].mode()[0])

# Limpieza rápida por moda para nulos huérfanos que puedan aparecer solo en Test
for col in X_test.columns:
    if X_test[col].dtype == 'object' and X_test[col].isnull().any():
        X_test[col] = X_test[col].fillna(X_train[col].mode()[0])
    elif X_test[col].dtype != 'object' and X_test[col].isnull().any() and col != 'LotFrontage':
        X_test[col] = X_test[col].fillna(0) # Áreas o características numéricas vacías en test se asumen 0

# ------------------------------------------------------------------------------
# 4. AUSENCIA REAL CRÍTICA (LotFrontage) -> Mediana por Vecindario (Neighborhood)
# ------------------------------------------------------------------------------
# Calculamos las medianas de LotFrontage por vecindario usando solo Train
neighborhood_median = X_train.groupby('Neighborhood')['LotFrontage'].transform('median')

X_train['LotFrontage'] = X_train['LotFrontage'].fillna(neighborhood_median)
# Para Test, mapeamos las medianas calculadas en Train para evitar data leakage
X_test['LotFrontage'] = X_test['LotFrontage'].fillna(X_test['Neighborhood'].map(X_train.groupby('Neighborhood')['LotFrontage'].median()))

# Por si algún vecindario en test quedara huérfano (caso extremo), usamos la mediana global
X_train['LotFrontage'] = X_train['LotFrontage'].fillna(X_train['LotFrontage'].median())
X_test['LotFrontage'] = X_test['LotFrontage'].fillna(X_train['LotFrontage'].median())

# ------------------------------------------------------------------------------
# VERIFICACIÓN FINAL DEL PIPELINE
# ------------------------------------------------------------------------------
print("=== VERIFICACIÓN DE CONTROL DE NULOS ===")
print(f"Nulos remanentes en Train: {X_train.isnull().sum().sum()}")
print(f"Nulos remanentes en Test:  {X_test.isnull().sum().sum()}")
print("========================================")

# ==============================================================================
# FASE 2 — PREPROCESAMIENTO (DATA PIPELINE)
# Paso 2B: Encoding con Diccionario Central (Ordinal + One-Hot)
# ==============================================================================

# 1. Definición del Diccionario Centralizado de 5 niveles (más el "None" estructural)
quality_mapping = {
    'Ex': 5, # Excellent
    'Gd': 4, # Good
    'TA': 3, # Typical/Average
    'Fa': 2, # Fair
    'Po': 1, # Poor
    'None': 0 # No tiene la estructura
}

# Lista de variables que utilizan estrictamente esta escala de calidad/condición
ordinal_columns = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
    'HeatingQC', 'KitchenQual', 'FireplaceQu', 
    'GarageQual', 'GarageCond', 'PoolQC'
]

# 2. Aplicar el mapeo ordinal explícito en ambos DataFrames
for col in ordinal_columns:
    X_train[col] = X_train[col].map(quality_mapping)
    X_test[col] = X_test[col].map(quality_mapping)

# 3. Identificar las variables nominales restantes para One-Hot Encoding
# Son aquellas columnas que siguen siendo de tipo 'object' (texto)
nominal_columns = X_train.select_dtypes(include=['object']).columns.tolist()

# 4. Aplicar One-Hot Encoding usando pd.get_dummies
X_train_encoded = pd.get_dummies(X_train, columns=nominal_columns, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=nominal_columns, drop_first=True)

# ------------------------------------------------------------------------------
# VERIFICACIÓN DE ESTRUCTURAS POST-ENCODING
# ------------------------------------------------------------------------------
print("=== VERIFICACIÓN DE CODIFICACIÓN ===")
print(f"Dimensiones de X_train post-encoding: {X_train_encoded.shape}")
print(f"Dimensiones de X_test post-encoding:  {X_test_encoded.shape}")
print("========================================")

# ==============================================================================
# FASE 2 — PREPROCESAMIENTO (DATA PIPELINE)
# Paso 2C: Alineación de Matrices Post-Encoding
# ==============================================================================

# 1. Guardamos la variable objetivo (SalePrice) que solo existe en Train
# Al aplicar pd.log1p como definiste en tu Paso 1A
y_train = np.log1p(train['SalePrice'])

# 2. Alinear los DataFrames para que compartan exactamente las mismas columnas.
# El argumento 'join="left"' conserva las columnas de Train y descarta las de Test que no sirvan.
X_train_final, X_test_final = X_train_encoded.align(X_test_encoded, join='left', axis=1)

# 3. Red de seguridad: Las columnas de Train que no existían en Test se llenarán con NaN tras el align.
# Como son columnas binarias (dummies) producto del One-Hot, lo correcto es rellenarlas con 0.
X_test_final = X_test_final.fillna(0)

# ------------------------------------------------------------------------------
# VERIFICACIÓN DE CONTROL ABSOLUTO
# ------------------------------------------------------------------------------
print("=== ALINEACIÓN COMPLETADA CON ÉXITO ===")
print(f"Dimensiones finales de X_train: {X_train_final.shape}")
print(f"Dimensiones finales de X_test:  {X_test_final.shape}")
print(f"¿Tienen exactamente las mismas columnas?: {list(X_train_final.columns) == list(X_test_final.columns)}")
print("========================================")

## Fase 3: Validación y Baseline (Ridge Regression)
Establecemos un punto de referencia (*baseline*) utilizando una regresión **Ridge**.

* **Regularización:** Utilizamos Ridge para controlar la complejidad y prevenir el sobreajuste (*overfitting*) en presencia de múltiples variables categóricas.
* **Estrategia de Evaluación:** Implementación de *K-Fold Cross-Validation* (5 pliegues) con mezcla de datos (*shuffle*), lo cual nos permite obtener una estimación sólida y no sesgada del error cuadrático medio (RMSE) antes de proceder con modelos más complejos.

In [ ]:
# ==============================================================================
# FASE 3 — VALIDACIÓN Y BASELINE
# Paso 3A: Configuración de K-Fold Cross-Validation
# ==============================================================================
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

# 1. Configurar K-Fold (5 pliegues para un balance óptimo entre sesgo y varianza)
# Shuffle=True es obligatorio para mezclar las casas y evitar sesgos por el orden de filas
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Instanciar nuestro modelo base (Regresión Ridge para controlar la complejidad)
# Usamos un alpha=10.0 inicial como buena práctica de regularización
ridge_baseline = Ridge(alpha=10.0)

# 3. Calcular el Root Mean Squared Error (RMSE) mediante Cross-Validation
# Usamos 'neg_root_mean_squared_error' porque scikit-learn maximiza las métricas por defecto
scores = cross_val_score(ridge_baseline, X_train_final, y_train, 
                         scoring='neg_root_mean_squared_error', cv=kf)

# 4. Convertir los scores a valores positivos
rmse_scores = -scores

print("=== RESULTADOS DEL BASELINE RIDGE ===")
print(f"RMSE por cada Fold: {np.round(rmse_scores, 4)}")
print(f"RMSE Promedio Global: {rmse_scores.mean():.4f}")
print(f"Desviación Estándar (Estabilidad): {rmse_scores.std():.4f}")
print("=====================================")

## Fase 4: Ingeniería de Características (Feature Engineering)
Potenciamos el poder predictivo del modelo mediante la creación de nuevas variables basadas en el dominio del problema.

* **Dimensión Espacial:** Agregación de áreas (sótano, 1er piso, 2do piso) para obtener una medida de superficie total habitable (`Total_SF`).
* **Dimensión Temporal:** Cálculo de la edad de la vivienda y años desde la última remodelación al momento de la venta.
* **Saneamiento:** Detección y eliminación de valores atípicos (*outliers*) críticos y control de multicolinealidad para eliminar redundancias que puedan desestabilizar los coeficientes del modelo.

In [ ]:
# ==============================================================================
# FASE 4 — FEATURE ENGINEERING
# Paso 4A: Creación de Variables de Dominio (Espacio y Tiempo)
# ==============================================================================

# Nota: Creamos las variables sobre las matrices ya alineadas finales para expandir el conocimiento del modelo

# 1. Dimensión Espacial (Superficie Total Habitable y de Servicio)
# Combinamos el área del sótano, el primer piso y el segundo piso en una métrica macro
X_train_final['Total_SF'] = X_train_final['TotalBsmtSF'] + X_train_final['1stFlrSF'] + X_train_final['2ndFlrSF']
X_test_final['Total_SF'] = X_test_final['TotalBsmtSF'] + X_test_final['1stFlrSF'] + X_test_final['2ndFlrSF']

# 2. Dimensión Temporal (Ciclo de Vida de la Estructura)
# Calculamos los años transcurridos desde la construcción original hasta la venta
X_train_final['YearsSinceBuilt'] = X_train_final['YrSold'] - X_train_final['YearBuilt']
X_test_final['YearsSinceBuilt'] = X_test_final['YrSold'] - X_test_final['YearBuilt']

# Calculamos los años transcurridos desde la última remodelación registrada hasta la venta
X_train_final['YearsSinceRemod'] = X_train_final['YrSold'] - X_train_final['YearRemodAdd']
X_test_final['YearsSinceRemod'] = X_test_final['YrSold'] - X_test_final['YearRemodAdd']

# 3. Red de seguridad para años negativos (por desfases atípicos menores en el registro de fechas)
for df in [X_train_final, X_test_final]:
    df['YearsSinceBuilt'] = df['YearsSinceBuilt'].clip(lower=0)
    df['YearsSinceRemod'] = df['YearsSinceRemod'].clip(lower=0)

print("=== INGENIERÍA DE CARACTERÍSTICAS COMPLETADA ===")
print(f"Nuevas dimensiones de X_train_final: {X_train_final.shape}")
print(f"Nuevas dimensiones de X_test_final:  {X_test_final.shape}")
print("==================================================")

# ==============================================================================
# FASE 4 — FEATURE ENGINEERING
# Paso 4B: Limpieza Avanzada (Outliers y Multicolinealidad)
# ==============================================================================

print("=== INICIANDO PASO 4B: LIMPIEZA AVANZADA ===")

# ------------------------------------------------------------------------------
# 1. REMOCIÓN DE OUTLIERS (Sobre las matrices de la Fase 4)
# ------------------------------------------------------------------------------
# Aplicamos el criterio metodológico sobre la variable original mapeada a los índices actuales
outlier_indices = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index

# Eliminamos las filas correspondientes de nuestras matrices de entrenamiento finales
X_train_final = X_train_final.drop(outlier_indices, errors='ignore').reset_index(drop=True)
y_train = y_train.drop(outlier_indices, errors='ignore').reset_index(drop=True)

print(f"  -> Outliers removidos del set de entrenamiento: {len(outlier_indices)}")

# ------------------------------------------------------------------------------
# 2. CONTROL DE MULTICOLINEALIDAD TRAS FEATURE ENGINEERING
# ------------------------------------------------------------------------------
# Calculamos la matriz de correlación solo sobre las variables explicativas (X_train)
corr_matrix = X_train_final.corr().abs()

# Seleccionamos el triángulo superior de la matriz de correlación para evitar evaluar pares duplicados
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Identificamos columnas con una correlación superior al 95% (Redundancia crítica)
# Esto removerá sutilmente variables duplicadas exactas generadas en el One-Hot o combinaciones espaciales
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

# Eliminamos las columnas redundantes de forma simétrica en Train y Test
X_train_final = X_train_final.drop(columns=to_drop)
X_test_final = X_test_final.drop(columns=to_drop)

print(f"  -> Columnas colineales eliminadas (>95% correlación): {len(to_drop)}")
print(f"     Variables descartadas: {to_drop}")

print("\n=== SANEAMIENTO MULTIVARIANTE COMPLETADO ===")
print(f"Dimensiones definitivas de X_train_final: {X_train_final.shape}")
print(f"Dimensiones definitivas de X_test_final:  {X_test_final.shape}")
print(f"¿Persiste la simetría exacta entre conjuntos?: {list(X_train_final.columns) == list(X_test_final.columns)}")
print("==============================================================")

# ==============================================================================
# CONTROL DE CALIDAD — EVALUACIÓN DEL PASO 4B
# ==============================================================================

# Volvemos a calcular el Root Mean Squared Error (RMSE) con las mismas reglas
scores_actualizado = cross_val_score(ridge_baseline, X_train_final, y_train, 
                                     scoring='neg_root_mean_squared_error', cv=kf)

rmse_actualizado = -scores_actualizado

print("=== NUEVOS RESULTADOS TRAS LIMPIEZA (PASO 4B) ===")
print(f"Nuevo RMSE por cada Fold: {np.round(rmse_actualizado, 4)}")
print(f"Nuevo RMSE Promedio Global: {rmse_actualizado.mean():.4f}")
print(f"Nueva Desviación Estándar:  {rmse_actualizado.std():.4f}")
print("=================================================")

## Fase 5: Modelado Avanzado y Predicción
En la etapa final, implementamos una arquitectura de ensamble (*Stacking*) para capitalizar la capacidad de aprendizaje de diferentes algoritmos.

* **Modelos Base:** XGBoost y LightGBM (Gradient Boosting) se combinan con nuestro baseline de Ridge para capturar relaciones lineales y no lineales en el dataset.
* **Meta-Modelo:** Utilizamos una regresión Ridge con validación cruzada interna para integrar las predicciones de los modelos base, minimizando el error general del ensamble.
* **Inferencia:** Reversión de la transformación logarítmica (`np.expm1`) para obtener las predicciones en la escala de dólares originales y generación del archivo final de sumisión para Kaggle.

In [ ]:
# ==============================================================================
# FASE 5 — MODELADO AVANZADO Y PREDICCIÓN
# Paso 5A: Configuración y Validación de Gradient Boosting (XGBoost + LightGBM)
# ==============================================================================
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("=== INICIANDO FASE 5A: GRADIENT BOOSTING ===")

# ------------------------------------------------------------------------------
# 1. Instanciar Modelos con Hiperparámetros de Regularización Avanzada
# ------------------------------------------------------------------------------
# XGBoost: Controlamos la complejidad del árbol con max_depth y subsample
xgb_model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

# LightGBM: Algoritmo por hojas ultra rápido. Controlamos con num_leaves
lgbm_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1,
    verbose=-1 # Silenciar advertencias de inicialización
)

# ------------------------------------------------------------------------------
# 2. Evaluación por Validación Cruzada (K-Fold)
# ------------------------------------------------------------------------------
# Usamos el mismo objeto 'kf' (5 pliegues) creado en la Fase 3 para comparar peras con peras
print("\n[Evaluando XGBoost...]")
xgb_scores = cross_val_score(xgb_model, X_train_final, y_train, scoring='neg_root_mean_squared_error', cv=kf)
xgb_rmse = -xgb_scores

print("[Evaluando LightGBM...]")
lgbm_scores = cross_val_score(lgbm_model, X_train_final, y_train, scoring='neg_root_mean_squared_error', cv=kf)
lgbm_rmse = -lgbm_scores

# ------------------------------------------------------------------------------
# 3. Reporte Comparativo de Rendimiento Local
# ------------------------------------------------------------------------------
print("\n==================================================")
print("             AUDITORÍA DE MODELOS EN 5A")
print("==================================================")
print(f"BASELINE RIDGE (Fase 3/4):")
print(f"  -> RMSE Promedio Global: {rmse_actualizado.mean():.4f}")
print("-" * 50)
print(f"XGBOOST REGRESSOR:")
print(f"  -> RMSE por cada Fold:   {np.round(xgb_rmse, 4)}")
print(f"  -> RMSE Promedio Global: {xgb_rmse.mean():.4f} (Estabilidad: {xgb_rmse.std():.4f})")
print("-" * 50)
print(f"LIGHTGBM REGRESSOR:")
print(f"  -> RMSE por cada Fold:   {np.round(lgbm_rmse, 4)}")
print(f"  -> RMSE Promedio Global: {lgbm_rmse.mean():.4f} (Estabilidad: {lgbm_rmse.std():.4f})")
print("==================================================")

# ==============================================================================
# FASE 5 — MODELADO AVANZADO Y PREDICCIÓN
# Paso 5B: Configuración y Validación de Stacking (Meta-Modelo)
# ==============================================================================
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

print("=== INICIANDO FASE 5B: STACKING REGRESSOR ===")

# ------------------------------------------------------------------------------
# 1. Definir los Modelos Base (Los especialistas que ya evaluamos)
# ------------------------------------------------------------------------------
base_models = [
    ('ridge', ridge_baseline),
    ('xgb', xgb_model),
    ('lgbm', lgbm_model)
]

# ------------------------------------------------------------------------------
# 2. Configurar el Meta-Modelo (El juez que combinará las predicciones)
# ------------------------------------------------------------------------------
# Usamos una Regresión Ridge con Cross-Validation interno (RidgeCV) como Meta-Estimador
# para que asigne los pesos óptimos a cada modelo base sin sobreajustar.
stacking_meta = StackingRegressor(
    estimators=base_models,
    final_estimator=RidgeCV(alphas=np.logspace(-3, 3, 10)),
    cv=kf,       # Validación cruzada interna para generar las meta-características
    n_jobs=-1
)

# ------------------------------------------------------------------------------
# 3. Evaluación del Ensamble General con K-Fold
# ------------------------------------------------------------------------------
print("\n[Entrenando y evaluando el Stacking Regressor...]")
stacking_scores = cross_val_score(stacking_meta, X_train_final, y_train, 
                                   scoring='neg_root_mean_squared_error', cv=kf)
stacking_rmse = -stacking_scores

# ------------------------------------------------------------------------------
# 4. Reporte Final de la Competencia Local
# ------------------------------------------------------------------------------
print("\n==================================================")
print("             VEREDICTO FINAL DE LA FASE 5")
print("==================================================")
print(f"RIDGE BASELINE (Lineal):  {rmse_actualizado.mean():.4f}")
print(f"XGBOOST (No Lineal):      {xgb_rmse.mean():.4f}")
print(f"LIGHTGBM (No Lineal):     {lgbm_rmse.mean():.4f}")
print("-" * 50)
print(f"STACKING REGRESSOR (Ensamble):")
print(f"  -> RMSE por cada Fold:   {np.round(stacking_rmse, 4)}")
print(f"  -> RMSE Promedio Global: {stacking_rmse.mean():.4f} (Estabilidad: {stacking_rmse.std():.4f})")
print("==================================================")

# ==============================================================================
# FASE 5 — MODELADO AVANZADO Y PREDICCIÓN
# Paso 5C: Entrenamiento Final y Generación de la Sumisión para Kaggle (Sin re-lectura)
# ==============================================================================
print("=== INICIANDO PASO 5C: ENTRENAMIENTO FINAL Y SUMISIÓN ===")

# 1. Ajustar el Meta-Modelo definitivo con TODO el set de entrenamiento
print("[1/3] Entrenando el Stacking Regressor definitivo con el 100% de los datos...")
stacking_meta.fit(X_train_final, y_train)

# 2. Realizar las predicciones sobre el set de prueba oficial
print("[2/3] Generando predicciones sobre el set de test...")
log_predictions = stacking_meta.predict(X_test_final)

# 3. Revertir la transformación logarítmica (Pasar de log a Dólares reales)
final_prices = np.expm1(log_predictions)

# 4. Intentar recuperar los IDs de las variables en memoria
# Probamos las tres formas más comunes en las que se suelen guardar al inicio:
if 'test_ids' in locals() or 'test_ids' in globals():
    ids_finales = test_ids
elif 'test' in locals() and 'Id' in test.columns:
    ids_finales = test['Id']
else:
    # Si por alguna razón no está en memoria, forzamos la búsqueda en el sistema
    import os
    import pandas as pd
    posibles_rutas = [
        '../input/house-prices-advanced-regression-techniques/test.csv',
        'test.csv'
    ]
    ruta_correcta = None
    for r in posibles_rutas:
        if os.path.exists(r):
            ruta_correcta = r
            break
    
    if ruta_correcta:
        ids_finales = pd.read_csv(ruta_correcta, usecols=['Id'])['Id']
    else:
        raise ValueError("No se encontraron los IDs en memoria ni el archivo test.csv. Verifica si la competencia está unida al notebook.")

# 5. Construir y exportar el DataFrame de sumisión estructurado
submission = pd.DataFrame({
    'Id': ids_finales,
    'SalePrice': final_prices
})

submission.to_csv('submission.csv', index=False)
print("[3/3] ¡Archivo 'submission.csv' generado con éxito en el directorio actual!")
print("-" * 50)
print(f"Total de registros a enviar: {submission.shape[0]}")
print(f"Rango de precios estimados:  ${submission['SalePrice'].min():,.2f} - ${submission['SalePrice'].max():,.2f}")
print("=========================================================")